# Full Waveform Inversion — Quickstart

End-to-end FWI in ~100 lines of Python, using [`deepwave`](https://github.com/ar4/deepwave) for differentiable wave-equation modeling in PyTorch.

**What this notebook does**

1. Builds a synthetic 2D velocity model with a buried high-velocity anomaly
2. Sets up a multi-shot surface acquisition (sources + receivers + Ricker wavelet)
3. Generates *observed* seismic data via the scalar acoustic wave equation
4. Starts from a poor initial guess (uniform velocity)
5. Recovers the true model by minimizing an L2 data misfit through **Adam**, with gradients computed end-to-end through the wave solver (automatic differentiation, no hand-written adjoint code)
6. Compares true / initial / FWI-inverted velocity panels

Runs on **CPU in ~2 minutes** at the default grid (100×100). Set `device = 'cuda'` for a 5–10× speedup if a GPU is available.

## 1. Imports and device

In [ ]:
import torch
import matplotlib.pyplot as plt
import deepwave
from deepwave import scalar

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(0)

## 2. Build the velocity models

We use a small 100 × 100 grid (∼cell size 4 m, total domain 400 × 400 m). The **true** model is a uniform 1500 m/s background with a buried 1900 m/s rectangular anomaly. The **initial** guess is uniform 1500 m/s — i.e., the inversion knows nothing about the anomaly's location, shape, or amplitude.

In [ ]:
ny, nx = 100, 100       # grid: (depth, horizontal)
dy = dx = 4.0            # cell size in meters

# True model: 1500 m/s background, buried 1900 m/s rectangular anomaly
v_true = torch.full((ny, nx), 1500.0)
v_true[45:65, 40:70] = 1900.0
v_true = v_true.to(device)

# Initial guess: uniform 1500 m/s (the inverter knows nothing)
v_init = torch.full((ny, nx), 1500.0, device=device)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, m, title in zip(axes, [v_true, v_init], ['True velocity', 'Initial guess']):
    im = ax.imshow(m.cpu(), cmap='viridis', vmin=1400, vmax=2000,
                   extent=[0, nx*dx, ny*dy, 0])
    ax.set_title(title)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('depth (m)')
    plt.colorbar(im, ax=ax, label='v (m/s)', shrink=0.8)
plt.tight_layout()
plt.show()

## 3. Acquisition geometry

Five shots evenly spaced along the surface (top row, `y = 0`). Each shot is recorded by 50 receivers also on the surface. This is a classical surface seismic geometry; the same machinery works for transmission / ring-array ultrasound — only the source/receiver coordinates change.

**Source wavelet**: 25 Hz Ricker, peak at `1.5 / freq` seconds.

In [ ]:
n_shots = 5
n_sources_per_shot = 1
n_receivers_per_shot = 50

freq = 25.0           # Hz (peak frequency of Ricker)
dt = 0.004            # 4 ms time step
nt = 750              # total time steps -> 3.0 s
peak_time = 1.5 / freq

# Source positions: top row, evenly spaced in x
source_locations = torch.zeros(n_shots, n_sources_per_shot, 2, dtype=torch.long, device=device)
source_locations[:, 0, 0] = 1                                                # depth index = 1 (just below surface)
source_locations[:, 0, 1] = torch.linspace(10, nx - 11, n_shots).long()

# Receiver positions: top row, 50 receivers spanning the domain
receiver_locations = torch.zeros(n_shots, n_receivers_per_shot, 2, dtype=torch.long, device=device)
receiver_locations[:, :, 0] = 1
receiver_locations[:, :, 1] = (
    torch.linspace(2, nx - 3, n_receivers_per_shot).long().unsqueeze(0).repeat(n_shots, 1)
)

# Ricker wavelet, replicated across shots: shape (n_shots, n_sources_per_shot, nt)
source_amplitudes = (
    deepwave.wavelets.ricker(freq, nt, dt, peak_time)
    .reshape(1, 1, -1)
    .repeat(n_shots, n_sources_per_shot, 1)
    .to(device)
)

print(f'source_amplitudes shape: {tuple(source_amplitudes.shape)}')
print(f'source_locations    shape: {tuple(source_locations.shape)}')
print(f'receiver_locations  shape: {tuple(receiver_locations.shape)}')

## 4. Forward model: generate the observed data

Propagate the wavelet through the **true** velocity field; record receiver traces. This is what real instruments would measure — we use it as our inversion target.

In [ ]:
with torch.no_grad():
    out = scalar(v_true, dx, dt,
                 source_amplitudes=source_amplitudes,
                 source_locations=source_locations,
                 receiver_locations=receiver_locations,
                 pml_freq=freq)
    observed_data = out[-1]   # shape: (n_shots, n_receivers_per_shot, nt)

print(f'observed_data shape: {tuple(observed_data.shape)}')

# Visualize the middle shot gather
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
shot = observed_data[n_shots // 2].cpu().T
vmax = shot.abs().max().item()
ax.imshow(shot, cmap='seismic', vmin=-vmax, vmax=vmax,
          extent=[0, n_receivers_per_shot, nt*dt, 0], aspect='auto')
ax.set_xlabel('receiver index')
ax.set_ylabel('time (s)')
ax.set_title(f'Observed data — shot {n_shots // 2}')
plt.show()

## 5. FWI optimization loop

Make `v` a leaf tensor with `requires_grad=True`. At each iteration:

1. Forward-propagate from the current `v` to get predicted data
2. Compute L2 misfit against observed
3. `loss.backward()` — autograd traces through the entire wave solver and yields `v.grad`
4. Adam updates `v`

No hand-derived adjoint required — the differentiable solver handles it.

In [ ]:
v = v_init.clone().requires_grad_(True)
optimizer = torch.optim.Adam([v], lr=20.0)
n_iter = 120

losses = []
for it in range(n_iter):
    optimizer.zero_grad()
    out = scalar(v, dx, dt,
                 source_amplitudes=source_amplitudes,
                 source_locations=source_locations,
                 receiver_locations=receiver_locations,
                 pml_freq=freq)
    pred = out[-1]
    loss = 1e9 * ((pred - observed_data) ** 2).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if it % 10 == 0 or it == n_iter - 1:
        print(f'iter {it:3d}  loss = {loss.item():.4e}')

v_inv = v.detach()

## 6. Loss curve and final comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
ax.semilogy(losses)
ax.set_xlabel('iteration')
ax.set_ylabel('L2 data misfit (log)')
ax.set_title('FWI convergence')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
panels = [('True',    v_true),
          ('Initial', v_init),
          ('FWI',     v_inv)]
for ax, (title, m) in zip(axes, panels):
    im = ax.imshow(m.cpu(), cmap='viridis', vmin=1400, vmax=2000,
                   extent=[0, nx*dx, ny*dy, 0])
    ax.set_title(title)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('depth (m)')
    plt.colorbar(im, ax=ax, label='v (m/s)', shrink=0.8)
plt.tight_layout()
plt.show()

## What just happened

- The wave solver is **fully differentiable**: `loss.backward()` carries gradients all the way back to `v` through 750 time steps of finite-difference propagation per shot.
- The misfit function is plain L2 between observed and simulated receiver traces. No adjoint code was written by hand — `deepwave` registers a custom autograd `Function` that exposes the adjoint as `backward()`.
- The inversion recovers the anomaly's **position and shape** well. Velocity amplitude is partially recovered; longer training, a smaller learning rate, or multi-scale (start from a low-pass-filtered wavelet) would refine it further.

## Where to go next

- **Multi-scale FWI**: filter the source wavelet to a low band first; invert; raise the band. Avoids cycle-skipping on real data.
- **Box constraints**: keep `v ∈ [v_min, v_max]` by reparameterizing through a sigmoid.
- **Real-world geometry**: ring-array transmission for ultrasound CT, ocean-bottom nodes for marine seismic — change `source_locations` and `receiver_locations`, everything else is the same.
- **Attenuating media**: swap `scalar` for a viscoacoustic solver (Q-factor) when the data shows significant amplitude decay.
- **Source estimation**: jointly invert for `v` and the source wavelet; replace the fixed `source_amplitudes` with a tensor that also has `requires_grad=True`.

## References

- `deepwave` — Alan Richardson, https://github.com/ar4/deepwave
- Virieux, J., & Operto, S. (2009). *An overview of full-waveform inversion in exploration geophysics.* Geophysics, **74**(6).
- Tarantola, A. (1984). *Inversion of seismic reflection data in the acoustic approximation.* Geophysics, **49**(8).